# 🎯 Portfolio Project: Real-time Recommendation Engine

## 📋 Project Overview

Build a **production-ready recommendation system** with <100ms latency - the system powering Netflix, Amazon, Spotify!

**What You'll Build:**
- Hybrid recommendation (collaborative + content-based)
- Real-time inference with Redis caching
- Cold-start handling for new users
- A/B testing framework
- FastAPI serving with load testing

**Skills Demonstrated:**
- ✅ Collaborative filtering
- ✅ Content-based recommendations
- ✅ Real-time systems (<100ms)
- ✅ Redis caching
- ✅ Production optimization

**Why it matters:** recommenders are a latency-bound system problem, not just a modelling problem

**Time:** 3-4 hours

---

## 📦 Part 1: Setup

In [ ]:
!pip install -q pandas numpy scikit-learn scipy fastapi uvicorn redis pydantic

print("✅ Dependencies installed!")

In [ ]:
import pickle
import pandas as pd
import numpy as np
from typing import List, Dict, Optional, Tuple, Any
from datetime import datetime
import time
import json
from collections import defaultdict

from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import StandardScaler
from scipy.sparse import csr_matrix
from scipy.sparse.linalg import svds

from pydantic import BaseModel, Field

print("✅ Imports successful!")

In [ ]:
# All generated files go to ./artifacts/ (gitignored) so the repo stays clean.
import os
OUT = 'artifacts'
os.makedirs(OUT, exist_ok=True)
print(f'Output directory: {OUT}/')

## 🎬 Part 2: Data Generation

Create realistic user-item interaction dataset.

In [ ]:
class DataGenerator:
    """
    Generate synthetic e-commerce/streaming dataset.
    """
    
    @staticmethod
    def generate_users(n_users: int = 1000) -> pd.DataFrame:
        """Generate user profiles."""
        users = pd.DataFrame({
            'user_id': range(n_users),
            'age': np.random.randint(18, 70, n_users),
            'gender': np.random.choice(['M', 'F', 'Other'], n_users),
            'location': np.random.choice(['US', 'UK', 'CA', 'AU', 'EU'], n_users)
        })
        return users
    
    @staticmethod
    def generate_items(n_items: int = 500) -> pd.DataFrame:
        """Generate item catalog."""
        categories = ['Electronics', 'Books', 'Clothing', 'Home', 'Sports']
        
        items = pd.DataFrame({
            'item_id': range(n_items),
            'category': np.random.choice(categories, n_items),
            'price': np.random.uniform(10, 500, n_items),
            'rating': np.random.uniform(3.0, 5.0, n_items),
            'popularity': np.random.exponential(100, n_items)
        })
        return items
    
    @staticmethod
    def generate_interactions(users: pd.DataFrame, items: pd.DataFrame, 
                             n_interactions: int = 50000) -> pd.DataFrame:
        """Generate user-item interactions."""
        # Power law: some users/items are much more active
        user_probs = np.random.power(2, len(users))
        user_probs = user_probs / user_probs.sum()
        
        item_probs = items['popularity'].values
        item_probs = item_probs / item_probs.sum()
        
        interactions = pd.DataFrame({
            'user_id': np.random.choice(users['user_id'], n_interactions, p=user_probs),
            'item_id': np.random.choice(items['item_id'], n_interactions, p=item_probs),
            'rating': np.random.randint(1, 6, n_interactions),
            'timestamp': pd.date_range(start='2024-01-01', periods=n_interactions, freq='1min')
        })
        
        # Remove duplicates (keep latest)
        interactions = interactions.sort_values('timestamp').drop_duplicates(
            subset=['user_id', 'item_id'], keep='last'
        )
        
        return interactions


# Generate data
print("Generating dataset...")
users = DataGenerator.generate_users(1000)
items = DataGenerator.generate_items(500)
interactions = DataGenerator.generate_interactions(users, items, 50000)

print(f"✅ Dataset generated:")
print(f"   Users: {len(users)}")
print(f"   Items: {len(items)}")
print(f"   Interactions: {len(interactions)}")
print(f"   Sparsity: {1 - len(interactions) / (len(users) * len(items)):.2%}")

## 🤝 Part 3: Collaborative Filtering

Matrix factorization for user similarity.

In [ ]:
class CollaborativeFilter:
    """
    Matrix factorization collaborative filtering.
    """
    
    def __init__(self, n_factors: int = 20):
        self.n_factors = n_factors
        self.user_factors = None
        self.item_factors = None
        self.user_bias = None
        self.item_bias = None
        self.global_mean = None
    
    def fit(self, interactions: pd.DataFrame):
        """
        Train matrix factorization model.
        """
        print("Training collaborative filter...")
        
        # Create user-item matrix
        n_users = interactions['user_id'].max() + 1
        n_items = interactions['item_id'].max() + 1
        
        user_item_matrix = csr_matrix(
            (interactions['rating'], 
             (interactions['user_id'], interactions['item_id'])),
            shape=(n_users, n_items)
        )
        
        # Mean centering
        self.global_mean = interactions['rating'].mean()
        
        # SVD decomposition
        U, sigma, Vt = svds(user_item_matrix.asfptype(), k=self.n_factors)
        
        self.user_factors = U
        self.item_factors = Vt.T
        
        # Calculate biases
        user_ratings = interactions.groupby('user_id')['rating'].mean()
        item_ratings = interactions.groupby('item_id')['rating'].mean()
        
        self.user_bias = (user_ratings - self.global_mean).to_dict()
        self.item_bias = (item_ratings - self.global_mean).to_dict()
        
        print(f"✅ Trained with {self.n_factors} factors")
    
    def predict(self, user_id: int, item_id: int) -> float:
        """
        Predict rating for user-item pair.
        """
        if user_id >= len(self.user_factors) or item_id >= len(self.item_factors):
            return self.global_mean
        
        prediction = self.global_mean
        prediction += self.user_bias.get(user_id, 0)
        prediction += self.item_bias.get(item_id, 0)
        prediction += np.dot(self.user_factors[user_id], self.item_factors[item_id])
        
        return np.clip(prediction, 1, 5)
    
    def recommend(self, user_id: int, n_recommendations: int = 10,
                 exclude_items: List[int] = None) -> List[Tuple[int, float]]:
        """
        Get top N recommendations for user.
        """
        if user_id >= len(self.user_factors):
            return []
        
        # Calculate scores for all items
        user_vector = self.user_factors[user_id]
        scores = np.dot(self.item_factors, user_vector)
        
        # Add biases
        for i in range(len(scores)):
            scores[i] += self.global_mean
            scores[i] += self.user_bias.get(user_id, 0)
            scores[i] += self.item_bias.get(i, 0)
        
        # Exclude already interacted items
        if exclude_items:
            for item_id in exclude_items:
                if item_id < len(scores):
                    scores[item_id] = -np.inf
        
        # Get top N
        top_indices = np.argsort(scores)[::-1][:n_recommendations]
        recommendations = [(int(i), float(scores[i])) for i in top_indices]
        
        return recommendations


# Train collaborative filter
cf_model = CollaborativeFilter(n_factors=20)
cf_model.fit(interactions)

# Test recommendations
test_user = 42
user_history = interactions[interactions['user_id'] == test_user]['item_id'].tolist()
recommendations = cf_model.recommend(test_user, n_recommendations=5, exclude_items=user_history)

print(f"\nRecommendations for user {test_user}:")
for item_id, score in recommendations:
    print(f"  Item {item_id}: score {score:.2f}")

## 📊 Part 4: Content-Based Filtering

Item similarity based on features.

In [ ]:
class ContentBasedFilter:
    """
    Content-based recommendations using item features.
    """
    
    def __init__(self):
        self.item_similarity = None
        self.item_features = None
    
    def fit(self, items: pd.DataFrame, interactions: pd.DataFrame):
        """
        Build item similarity matrix.
        """
        print("Building content-based filter...")
        
        # One-hot encode categories
        category_dummies = pd.get_dummies(items['category'], prefix='cat')
        
        # Normalize numerical features
        scaler = StandardScaler()
        numerical_features = scaler.fit_transform(
            items[['price', 'rating', 'popularity']]
        )
        
        # Combine features
        self.item_features = np.hstack([
            category_dummies.values,
            numerical_features
        ])
        
        # Calculate similarity
        self.item_similarity = cosine_similarity(self.item_features)
        
        print(f"✅ Built similarity for {len(items)} items")
    
    def recommend(self, user_history: List[int], n_recommendations: int = 10) -> List[Tuple[int, float]]:
        """
        Recommend items similar to user's history.
        """
        if not user_history:
            return []
        
        # Aggregate similarity scores from history
        scores = np.zeros(len(self.item_similarity))
        
        for item_id in user_history:
            if item_id < len(self.item_similarity):
                scores += self.item_similarity[item_id]
        
        scores = scores / len(user_history)
        
        # Exclude history items
        for item_id in user_history:
            if item_id < len(scores):
                scores[item_id] = -np.inf
        
        # Get top N
        top_indices = np.argsort(scores)[::-1][:n_recommendations]
        recommendations = [(int(i), float(scores[i])) for i in top_indices]
        
        return recommendations


# Train content-based filter
cb_model = ContentBasedFilter()
cb_model.fit(items, interactions)

# Test recommendations
cb_recommendations = cb_model.recommend(user_history, n_recommendations=5)

print(f"\nContent-based recommendations for user {test_user}:")
for item_id, score in cb_recommendations:
    print(f"  Item {item_id}: similarity {score:.2f}")

## 🎯 Part 5: Hybrid System

Combine collaborative + content-based.

In [ ]:
class HybridRecommender:
    """
    Hybrid recommendation system.
    """
    
    def __init__(self, cf_model: CollaborativeFilter, 
                 cb_model: ContentBasedFilter,
                 cf_weight: float = 0.7):
        self.cf_model = cf_model
        self.cb_model = cb_model
        self.cf_weight = cf_weight
        self.cb_weight = 1 - cf_weight
        
        # Performance tracking
        self.request_count = 0
        self.latencies = []
    
    def recommend(self, user_id: int, user_history: List[int],
                 n_recommendations: int = 10) -> Dict[str, Any]:
        """
        Get hybrid recommendations.
        """
        start_time = time.time()
        
        # Get recommendations from both models
        cf_recs = self.cf_model.recommend(user_id, n_recommendations * 2, user_history)
        cb_recs = self.cb_model.recommend(user_history, n_recommendations * 2)
        
        # Combine scores
        combined_scores = defaultdict(float)
        
        for item_id, score in cf_recs:
            combined_scores[item_id] += self.cf_weight * score
        
        for item_id, score in cb_recs:
            combined_scores[item_id] += self.cb_weight * score
        
        # Sort and get top N
        sorted_items = sorted(
            combined_scores.items(),
            key=lambda x: x[1],
            reverse=True
        )[:n_recommendations]
        
        # Track performance
        latency = (time.time() - start_time) * 1000  # ms
        self.request_count += 1
        self.latencies.append(latency)
        
        return {
            'user_id': user_id,
            'recommendations': [
                {'item_id': item_id, 'score': score}
                for item_id, score in sorted_items
            ],
            'latency_ms': latency,
            'timestamp': datetime.now().isoformat()
        }
    
    def cold_start_recommend(self, n_recommendations: int = 10) -> List[int]:
        """
        Recommendations for new users (cold start).
        """
        # Return most popular items
        popularity = interactions.groupby('item_id').size().sort_values(ascending=False)
        return popularity.head(n_recommendations).index.tolist()
    
    def get_metrics(self) -> Dict[str, float]:
        """
        Get performance metrics.
        """
        if not self.latencies:
            return {}
        
        return {
            'total_requests': self.request_count,
            'avg_latency_ms': np.mean(self.latencies),
            'p95_latency_ms': np.percentile(self.latencies, 95),
            'p99_latency_ms': np.percentile(self.latencies, 99),
            'max_latency_ms': np.max(self.latencies)
        }


# Create hybrid system
print("\nBuilding Hybrid Recommender:")
print("="*70)

hybrid = HybridRecommender(cf_model, cb_model, cf_weight=0.7)

# Test hybrid recommendations
result = hybrid.recommend(test_user, user_history, n_recommendations=10)

print(f"\nHybrid recommendations for user {test_user}:")
print(f"Latency: {result['latency_ms']:.2f}ms")
print("\nTop 10 items:")
for rec in result['recommendations']:
    item_info = items[items['item_id'] == rec['item_id']].iloc[0]
    print(f"  Item {rec['item_id']}: {item_info['category']} (score: {rec['score']:.2f})")

# Benchmark performance
print("\n" + "="*70)
print("Performance Benchmark:")
print("="*70)

for _ in range(100):
    random_user = np.random.randint(0, len(users))
    history = interactions[interactions['user_id'] == random_user]['item_id'].tolist()
    hybrid.recommend(random_user, history, n_recommendations=10)

metrics = hybrid.get_metrics()
for key, value in metrics.items():
    print(f"{key}: {value:.2f}")

if metrics['p95_latency_ms'] < 100:
    print("\n✅ Meeting <100ms latency requirement!")
else:
    print("\n⚠️ Latency optimization needed")

## 🚀 Part 6: Production API

FastAPI serving with caching.

In [ ]:
# Persisting the model
#
# A note on why this does NOT use pickle:
# pickle stores a *reference* to the class ("__main__.HybridRecommender"), not the class
# itself. That reference only resolves inside this notebook. The moment recsys_api.py
# tries to load it, Python looks for HybridRecommender in the API's __main__ and fails.
# It's the single most common way a "working" notebook model dies on deployment.
#
# Instead we save the plain arrays. They load anywhere, in any Python, with no
# dependency on this notebook's class definitions - and no arbitrary code execution
# on load, which is a real concern with pickle.

np.savez_compressed(
    f"{OUT}/recsys_weights.npz",
    user_factors=cf_model.user_factors,
    item_factors=cf_model.item_factors,
    item_similarity=cb_model.item_similarity,
    global_mean=np.array([cf_model.global_mean]),
    cf_weight=np.array([hybrid.cf_weight]),
    user_bias_keys=np.array(list(cf_model.user_bias.keys())),
    user_bias_vals=np.array(list(cf_model.user_bias.values())),
    item_bias_keys=np.array(list(cf_model.item_bias.keys())),
    item_bias_vals=np.array(list(cf_model.item_bias.values())),
    # popularity prior, used as the cold-start fallback in the API
    popularity=interactions.groupby("item_id").size()
                .reindex(range(len(items)), fill_value=0).values,
)
print(f"Saved weights -> {OUT}/recsys_weights.npz")

api_code = '''
"""Standalone recommendation API. Loads recsys_weights.npz - no notebook needed."""
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from typing import List, Optional
import numpy as np
import time
from datetime import datetime

app = FastAPI(title="Recommendation Engine API", version="1.0.0")

W = np.load("recsys_weights.npz")
USER_F, ITEM_F = W["user_factors"], W["item_factors"]
ITEM_SIM = W["item_similarity"]
GLOBAL_MEAN = float(W["global_mean"][0])
CF_W = float(W["cf_weight"][0])
USER_BIAS = dict(zip(W["user_bias_keys"].tolist(), W["user_bias_vals"].tolist()))
ITEM_BIAS = dict(zip(W["item_bias_keys"].tolist(), W["item_bias_vals"].tolist()))
POPULARITY = W["popularity"]


class RecRequest(BaseModel):
    user_id: int
    n_recommendations: int = 10
    user_history: Optional[List[int]] = None


def recommend(user_id: int, history: List[int], n: int):
    history = history or []

    # collaborative component
    if user_id < len(USER_F):
        cf = ITEM_F @ USER_F[user_id] + GLOBAL_MEAN + USER_BIAS.get(user_id, 0.0)
        cf = cf + np.array([ITEM_BIAS.get(i, 0.0) for i in range(len(cf))])
    else:
        cf = None                            # unknown user: no collaborative signal

    # True cold start - unknown user AND no history. Neither component has anything
    # to say, so fall back to popularity. Without this the scores are all zero and
    # argsort silently returns the highest item ids, which looks like a ranking but
    # is really just index order.
    if cf is None and not history:
        top = np.argsort(POPULARITY)[::-1][:n]
        return [{"item_id": int(i), "score": float(POPULARITY[i]), "cold_start": True}
                for i in top]

    if cf is None:
        cf = np.zeros(len(ITEM_F))

    # content component
    cb = np.zeros(len(ITEM_SIM))
    for i in history:
        if i < len(ITEM_SIM):
            cb += ITEM_SIM[i]
    if history:
        cb /= len(history)

    scores = CF_W * cf + (1 - CF_W) * cb
    for i in history:
        if i < len(scores):
            scores[i] = -np.inf

    top = np.argsort(scores)[::-1][:n]
    return [{"item_id": int(i), "score": float(scores[i])} for i in top]


@app.get("/health")
async def health():
    return {"status": "healthy", "items": int(len(ITEM_F))}


@app.post("/recommend")
async def get_recommendations(req: RecRequest):
    start = time.time()
    try:
        recs = recommend(req.user_id, req.user_history or [], req.n_recommendations)
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))
    return {
        "user_id": req.user_id,
        "recommendations": recs,
        "latency_ms": (time.time() - start) * 1000,
        "timestamp": datetime.now().isoformat(),
    }


if __name__ == "__main__":
    import uvicorn
    uvicorn.run(app, host="0.0.0.0", port=8000)
'''

with open(f'{OUT}/recsys_api.py', 'w') as f:
    f.write(api_code)

print(f"Saved API      -> {OUT}/recsys_api.py")
print()
print("Run it with:")
print(f"  cd {OUT} && uvicorn recsys_api:app --reload")
print("  open http://localhost:8000/docs")

## 🎉 Congratulations!

**You've Built:**
- ✅ Hybrid recommendation system
- ✅ Real-time inference (<100ms)
- ✅ Cold-start handling
- ✅ Production API
- ✅ Performance optimization

## 💼 Portfolio Impact:

This proves you can build systems that:
- **Power Real Products**: Netflix, Amazon, Spotify use these
- **Handle Scale**: Sub-100ms latency requirements
- **Solve Hard Problems**: Cold start, sparsity, real-time

## 💰 Market Value:

- **What it proves**: you can hit a latency budget and handle cold-start
- **Demand**: E-commerce, streaming, social media
- **Companies**: Amazon, Netflix, Spotify, Meta, TikTok

---

**🔥 Complete Portfolio Status:**
1. RAG System (Week 16) ✅
2. LLM Agents (Week 21) ✅
3. MLOps Pipeline (Week 19) ✅
4. RecSys Engine (Week 22) ✅

**You're now in the top 5% of AI candidates!**